## Enriching stock market data using Open AI API 

<p align="center">
    <img src="images/nasdaq100.png" width="450">
</p>

The Nasdaq-100 is a stock market index made up of 101 equity securities issued by 100 of the largest non-financial companies listed on the Nasdaq stock exchange. It helps investors compare stock prices with previous prices to determine market performance.

In this project you are provided with two CSV files containing Nasdaq-100 stock information:
- _**nasdaq100_CA.csv**_: contains information about companies in the index such as symbol, name, etc. For this analysis, only companies headquartered in California have been selected.
- _**nasdaq100_price_change.csv**_: contains price changes per stock across periods including (but not limited to) one day, five days, one month, six months, one year, etc.

As an AI developer, you will leverage the OpenAI API to classify companies into sectors and produce a summary of sector and company performance for this year, for the companies in the index that are headquartered in California.

# CSV with Nasdaq-100 stock data

In this project, you have available two CSV files `nasdaq100_CA.csv` and `nasdaq100_price_change.csv`.

## nasdaq100_CA.csv

```py
symbol,name,headQuarter,dateFirstAdded,cik,founded
AAPL,Apple Inc.,"Cupertino, CA",,0000320193,1976-04-01
ABNB,Airbnb,"San Francisco, CA",,0001559720,2008-08-01
ADBE,Adobe Inc.,"San Jose, CA",,0000796343,1982-12-01
...
```

## nasdaq100_price_change.csv

```py
symbol,1D,5D,1M,3M,6M,ytd,1Y,3Y,5Y,10Y,max
AAPL,-1.7254,-8.30086,-6.20411,3.042,15.64824,42.99992,8.47941,60.96299,245.42031,976.99441,139245.53954
ABNB,2.1617,-2.21919,9.88336,19.43286,19.64241,68.66902,23.64013,-1.04347,-1.04347,-1.04347,-1.04347
ADBE,0.5409,-1.77817,9.16191,52.0465,38.01522,57.22723,21.96206,17.83037,109.05718,1024.69214,251030.66399
ADI,0.9291,-4.03352,2.58486,3.65887,5.01602,17.02062,8.09735,63.42847,92.81874,286.77518,26012.63736
...
```

In [10]:
# Start your code here!
import os
import pandas as pd
from openai import OpenAI

# Instantiate an API client
client = OpenAI()

# Continue coding here
# Use as many cells as you like
# Task 1: Load nasdaq100_CA.csv and add ytd column
nasdaq100_ca = pd.read_csv('nasdaq100_CA.csv')
price_change = pd.read_csv('nasdaq100_price_change.csv')
nasdaq100_ca = nasdaq100_ca.merge(price_change[['symbol', 'ytd']], on='symbol', how='left')

# Task 2: Use OpenAI to classify each stock into a sector
nasdaq = nasdaq100_ca.copy()

def classify_sector(company_name, symbol):
        response = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[
                                    {"role": "system", "content": "You are a financial analyst. Classify the given company into exactly one of these sectors: Technology, Consumer Cyclical, Industrials, Utilities, Healthcare, Communication, Energy, Consumer Defensive, Real Estate, Financial. Respond with only the sector name, nothing else."},
                                    {"role": "user", "content": f"Classify this company: {company_name} (ticker: {symbol})"}
                                ],
                    max_tokens=10
                )
        return response.choices[0].message.content.strip()

nasdaq['sector'] = nasdaq.apply(lambda row: classify_sector(row['name'], row['symbol']), axis=1)
nasdaq100_ca['sector'] = nasdaq['sector']
print(nasdaq[['symbol', 'name', 'sector', 'ytd']].head())

# Task 3: Use OpenAI for stock recommendations
nasdaq_info = nasdaq[['symbol', 'name', 'sector', 'ytd']].to_string(index=False)

rec_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
                    {"role": "system", "content": "You are a financial analyst providing investment recommendations."},
                    {"role": "user", "content": f"Based on the following Nasdaq-100 California companies with their sectors and year-to-date (YTD) performance, recommend the two best sectors and at least two companies per sector:\n\n{nasdaq_info}"}
                ],
        max_tokens=500
    )

stock_recommendations = rec_response.choices[0].message.content
print(stock_recommendations)                

  symbol               name             sector       ytd
0   AAPL         Apple Inc.         Technology  42.99992
1   ABNB             Airbnb  Consumer Cyclical  68.66902
2   ADBE         Adobe Inc.         Technology  57.22723
3   ADSK           Autodesk         Technology  10.02701
4   AMAT  Applied Materials         Technology  55.46366
Based on the year-to-date (YTD) performance data provided, it appears that the two best-performing sectors are **Technology** and **Consumer Cyclical**.

### Recommended Sectors and Companies

#### 1. **Technology Sector**
The Technology sector has shown exceptional performance this year, with many companies recording impressive growth.

- **Apple Inc. (AAPL)**: YTD performance of 42.99%
- **Nvidia (NVDA)**: YTD performance of 217.27%
- **Meta Platforms (META)**: YTD performance of 153.78%
- **Advanced Micro Devices (AMD)**: YTD performance of 82.46%
  
#### 2. **Consumer Cyclical Sector**
The Consumer Cyclical sector has also demonstrated strong per